In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/biohub_embryo.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))


In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/biohub_embryo.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))


In [ ]:
import os
BIOHUB_PRESET = 'dual_seed_near_balanced_center_confirmed_synthetic_gap'
BIOHUB_SCORE_AXIS = 'near-balanced shared detections with marginal synthetic-gap confirmation'

# Parameters for constrained lineage reconstruction.
os.environ["BIOHUB_OUTPUT_FILTER_SHORT_TRACKS"] = "1"
os.environ["BIOHUB_DET_THRESHOLD"] = "0.96875"
os.environ["BIOHUB_MOTION_RELINK_LEARNED_BONUS"] = '1.0'
os.environ["BIOHUB_ILP_APPEARANCE_WEIGHT"] = "0.0"
os.environ["BIOHUB_ILP_DISAPPEARANCE_WEIGHT"] = "1.5"
os.environ["BIOHUB_GAP_CLOSE_MAX_GAP"] = "2"
os.environ["BIOHUB_GAP_CLOSE_UM"] = "5.8"
os.environ["BIOHUB_GAP_DENSITY_ADAPTIVE"] = "1"
os.environ["BIOHUB_GAP_DENSITY_REFERENCE_UM"] = "6.5"
os.environ["BIOHUB_GAP_DENSITY_GAIN"] = "0.040"
os.environ["BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM"] = "0.125"
os.environ["BIOHUB_GAP_DENSITY_NEIGHBORS"] = "3"
os.environ["BIOHUB_OUTPUT_MIN_TRACK_LEN"] = "6"
os.environ["BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS"] = "1"
os.environ["BIOHUB_OUTPUT_GAP2_RECOVERY"] = "0"
os.environ["BIOHUB_SAFE_DIV_MAX_UM"] = "4.66"
os.environ["BIOHUB_SAFE_DIV_SISTER_MAX_UM"] = '8.5'
os.environ["BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM"] = "7.65"
os.environ["BIOHUB_SAFE_DIV_FRAME_FRAC_CAP"] = "0.0076"
os.environ["BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP"] = "0.00375"
os.environ["BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE"] = "0"
os.environ["BIOHUB_USE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_REQUIRE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_EXPECTED_EPOCH"] = "500"
os.environ["BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM"] = "8.5"
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/checkpoint_last.pt"
os.environ["BIOHUB_DEEPCENTER_GAP_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_GAP_THRESHOLD"] = "0.25"
os.environ["BIOHUB_DEEPCENTER_SAFE_DIV_VETO"] = '0'
os.environ["BIOHUB_RUN_OUTPUT_DIAGNOSTICS"] = "0"

print("BIOHUB_PRESET:", BIOHUB_PRESET)
print("BIOHUB_SCORE_AXIS:", BIOHUB_SCORE_AXIS)

# --- factory env overrides ---
os.environ["H1R_KERNEL"] = '1'
os.environ["H1R_EPOCHS"] = '20'
os.environ["H1R_BS"] = '16'
os.environ["H1R_LR"] = '1e-4'
os.environ["H1R_NEG_WEIGHT"] = '0.1'
os.environ["H1R_MAX_STEPS"] = '0'
os.environ["H1R_VAL_MOD"] = '6'
os.environ["H1R_AMP"] = '1'
os.environ["H1R_BENCH_STEPS"] = '50'
os.environ["H1R_MIN_AMP_SPEEDUP"] = '1.3'
os.environ["H1R_EVAL_TTA"] = '1'
os.environ["H1R_SELECT_THRESHOLD"] = 'deployed'
os.environ["H1R_TRUNK_CONTRACT"] = 'freeze'
os.environ["H1R_REQUIRE_OPT_STEPS"] = '1'
os.environ["H1R_PATIENCE"] = '3'
os.environ["H1R_MIN_DELTA"] = '1e-4'
os.environ["H1R_EVAL_PROBS"] = '0.50,0.75,0.90,0.95,0.96875,0.98,0.99'
os.environ["H1R_ROT90"] = '1'
os.environ["H1R_RESUME"] = '0'
os.environ["H1R_WORKERS"] = '3'
os.environ["H1R_SEED"] = '0'


In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
_test_dir_override = os.environ.get("BIOHUB_TEST_DIR", "").strip()
TEST_DIR = Path(_test_dir_override) if _test_dir_override else COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"

METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
EXPERIMENT_TAG = "selected_101_dual_seed_near_balanced_center_confirmed_synthetic_gap"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"

DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))

# Empty for a real submission. Useful for local smoke tests, e.g. BIOHUB_SLICE=:1.
SLICE = os.environ.get("BIOHUB_SLICE", "").strip()

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))
GAP_DENSITY_ADAPTIVE = os.environ.get("BIOHUB_GAP_DENSITY_ADAPTIVE", "0") != "0"
GAP_DENSITY_REFERENCE_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_REFERENCE_UM", "6.5"))
GAP_DENSITY_GAIN = float(os.environ.get("BIOHUB_GAP_DENSITY_GAIN", "0.040"))
GAP_DENSITY_MAX_STEP_DELTA_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM", "0.125"))
GAP_DENSITY_NEIGHBORS = int(os.environ.get("BIOHUB_GAP_DENSITY_NEIGHBORS", "3"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "6"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"
ADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get("BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE", "0") != "0"
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC", "0.10"))
SHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN", "4"))
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB", "0.82"))
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM", "3.25"))
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC", "0.018"))
SHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS", "180"))

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "7.2"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.8"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))

# DeepCenter support is retained for compatibility, but this selected run keeps it disabled.
USE_DEEPCENTER_VETO = os.environ.get("BIOHUB_USE_DEEPCENTER_VETO", "1") != "0"
REQUIRE_DEEPCENTER_VETO = os.environ.get("BIOHUB_REQUIRE_DEEPCENTER_VETO", "1") != "0"
DEEPCENTER_MANIFEST_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",
)
DEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT",
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/checkpoint_last.pt",
)
DEEPCENTER_RELATIVE = os.environ.get("BIOHUB_DEEPCENTER_RELATIVE", "weights/full_frame_center/checkpoint_last.pt")
DEEPCENTER_GAP_VETO = os.environ.get("BIOHUB_DEEPCENTER_GAP_VETO", "1") != "0"
DEEPCENTER_SAFE_DIV_VETO = os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_VETO", "1") != "0"
DEEPCENTER_GAP_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_THRESHOLD", "0.10"))
DEEPCENTER_EXPECTED_EPOCH = int(os.environ.get("BIOHUB_DEEPCENTER_EXPECTED_EPOCH", "0"))
DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM", "0"))
DEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD", "0.12"))
DEEPCENTER_SCORE_WIN_Z = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_Z", "1"))
DEEPCENTER_SCORE_WIN_YX = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_YX", "2"))
DEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES", "8"))

CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_density_adaptive": GAP_DENSITY_ADAPTIVE,
    "gap_density_reference_um": GAP_DENSITY_REFERENCE_UM,
    "gap_density_gain": GAP_DENSITY_GAIN,
    "gap_density_max_step_delta_um": GAP_DENSITY_MAX_STEP_DELTA_UM,
    "gap_density_neighbors": GAP_DENSITY_NEIGHBORS,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "adaptive_short_track_rescue": ADAPTIVE_SHORT_TRACK_RESCUE,
    "short_track_rescue_trigger_removed_frac": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,
    "short_track_rescue_min_len": SHORT_TRACK_RESCUE_MIN_LEN,
    "short_track_rescue_min_mean_edge_prob": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,
    "short_track_rescue_max_mean_edge_dist_um": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,
    "short_track_rescue_max_nodes_frac": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,
    "short_track_rescue_max_nodes_abs": SHORT_TRACK_RESCUE_MAX_NODES_ABS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
    "use_deepcenter_add_only_gate": USE_DEEPCENTER_VETO,
    "deepcenter_gap_add_gate": DEEPCENTER_GAP_VETO,
    "deepcenter_safe_div_add_gate": DEEPCENTER_SAFE_DIV_VETO,
    "deepcenter_gap_threshold": DEEPCENTER_GAP_THRESHOLD,
    "deepcenter_expected_epoch": DEEPCENTER_EXPECTED_EPOCH,
    "deepcenter_gap_confirm_min_span_um": DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM,
    "deepcenter_safe_div_threshold": DEEPCENTER_SAFE_DIV_THRESHOLD,
    "deepcenter_checkpoint_default": DEEPCENTER_CHECKPOINT_DEFAULT,
}

print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))
    _expected_primary_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
    _actual_primary_sha256 = str(manifest_info.get("model", {}).get("weight_sha256", ""))
    if _actual_primary_sha256 != _expected_primary_sha256:
        raise RuntimeError(
            "Primary model checksum mismatch: "
            f"expected {_expected_primary_sha256}, got {_actual_primary_sha256 or 'missing'}"
        )

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)


# Resolve the independent-seed pack by checksum, then materialize only its weights.
import hashlib as _hashlib

_secondary_manifest_explicit = Path(os.environ.get(
    "BIOHUB_SECONDARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/ARTIFACT_MANIFEST.json",
))
_secondary_expected_sha256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
_secondary_slug = "biohub-temporal-unet3d-seed314159-v1"


def _find_secondary_artifact_root() -> tuple[Path, dict]:
    candidates = [
        _secondary_manifest_explicit,
        Path(f"/kaggle/input/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
        Path(f"/kaggle/input/datasets/pilkwang/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(input_root.rglob("ARTIFACT_MANIFEST.json"))

    seen = set()
    for manifest_path in candidates:
        manifest_path = manifest_path.expanduser()
        if manifest_path in seen or not manifest_path.is_file():
            continue
        seen.add(manifest_path)
        try:
            info = json.loads(manifest_path.read_text())
        except Exception:
            continue
        sha256 = str(info.get("model", {}).get("weight_sha256", ""))
        if sha256 == _secondary_expected_sha256:
            return manifest_path.parent, info
    raise FileNotFoundError(
        "Could not find the independent-seed artifact with weight SHA256 "
        + _secondary_expected_sha256
    )


SECONDARY_ARTIFACTS, secondary_manifest_info = _find_secondary_artifact_root()
SECONDARY_WEIGHTS_ROOT = WORKING_DIR / "secondary_seed_weights"
copy_or_extract_tree(
    SECONDARY_ARTIFACTS / "weights",
    SECONDARY_ARTIFACTS / "weights.zip",
    SECONDARY_WEIGHTS_ROOT,
)
SECONDARY_WEIGHTS_PATH = (
    SECONDARY_WEIGHTS_ROOT
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)
SECONDARY_CONFIG_PATH = SECONDARY_WEIGHTS_PATH.parent / "config.json"
for _required_secondary_path in (SECONDARY_WEIGHTS_PATH, SECONDARY_CONFIG_PATH):
    if not _required_secondary_path.is_file():
        raise FileNotFoundError(f"Missing secondary model file: {_required_secondary_path}")


def _sha256_file(path: Path) -> str:
    digest = _hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


_secondary_actual_sha256 = _sha256_file(SECONDARY_WEIGHTS_PATH)
if _secondary_actual_sha256 != _secondary_expected_sha256:
    raise RuntimeError(
        "Secondary model checksum mismatch: "
        f"expected {_secondary_expected_sha256}, got {_secondary_actual_sha256}"
    )

os.environ["BIOHUB_SECONDARY_WEIGHTS"] = str(SECONDARY_WEIGHTS_PATH)
os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"] = "0.15"
print("Secondary artifact:", SECONDARY_ARTIFACTS)
print("Secondary weight:", SECONDARY_WEIGHTS_PATH)
print("Secondary SHA256:", _secondary_actual_sha256)
print("Secondary edge-logit weight:", os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"])

os.environ["BIOHUB_SECONDARY_DETECTION_WEIGHT"] = "0.475"
os.environ["BIOHUB_SECONDARY_LINK_MODE"] = "low_margin_consensus"
os.environ["BIOHUB_SECONDARY_MIX_TEMPERATURE"] = "1"
os.environ["BIOHUB_SECONDARY_LOW_MARGIN_MAX"] = "0.35"
os.environ["BIOHUB_DUAL_SEED_EDGE_THRESHOLD"] = "0.48"


In [ ]:
r"""H1-R trainer patch -- exact-string fixes to the vendored training script.

Applies the retrain-recipe fixes (retrain_recipes_2026-08-17: A1/A3/A5/B4 + C2 hook) to a COPY
of ``train_unet_transformer.py`` (in a kernel: ``REPO_DIR/scripts/``; locally: a scratch copy --
never the vendored original in-place under ``vendor/``).

Every patch asserts an exact occurrence count, mirroring the kaggle_factory edit discipline:
a silently-skipped patch is an error, not a warning.

WHAT IT PATCHES (all locations verified against the vendored file 2026-08-18)
-----------------------------------------------------------------------------
P1  AMP (F5, trainer :64):  `F.binary_cross_entropy` raises under autocast, so autocast is
    applied INSIDE `TemporalUNet3D.forward` (monkeypatch appended to the module) with the
    output cast back to float32. This is deliberate: `nn.DataParallel` runs replica forwards
    in worker threads and `torch.autocast` is thread-local, so an autocast context around
    `model.encode(...)` would silently NOT autocast the 2-GPU path. Patching the UNet forward
    autocasts in every replica thread; everything downstream (detect head, transformer, both
    losses) stays fp32-safe. A `GradScaler` is added to `train_epoch`.
P2  Precision-aware checkpoint selection (F2, trainer :1180): `evaluate` additionally returns
    node_precision (matched detections / all detections); `train` selects on
    `test_acc * detection_F1` (env `H1R_SELECT=acc_f1`, default) or the legacy
    `test_acc * test_recall` (`H1R_SELECT=acc_recall`).
P3  Real division upweight (F6, trainer :68-70): `weight[div_rows] = _H1R_DIV_WEIGHT`
    (env `H1R_DIV_WEIGHT`, default 3.0; Trackastra uses 11).
P4  Sparse-annotation ignore mask (recipe C2) in `compute_detection_loss`: zero the negative
    weight in a ball of radius `H1R_IGNORE_RADIUS_VOX` voxels around confident non-GT peaks
    (likely unannotated true nuclei). Default 0 = off; enable ONLY for the competition
    fine-tune stage, never for dense Zebrahub labels.
P5  Cosine LR decay over epochs (recipe B4), env `H1R_COSINE` (default on).
P6  Always-save `edge_predictor_last.pth` each epoch for cross-session resume.

Usage (local test / kernel cell):
  .venv\Scripts\python.exe scripts\kaggle_edits\h1r_trainer_patch.py --trainer <path-to-copy>
  # or, in a notebook cell where this file's content was embedded:
  apply_h1r_trainer_patch(REPO_DIR / "scripts" / "train_unet_transformer.py")
"""
from __future__ import annotations

from pathlib import Path

_KNOBS = '''from itertools import cycle as _cycle

import os as _os

# --- H1-R patch knobs (scripts/kaggle_edits/h1r_trainer_patch.py) -------------------
_H1R_AMP = _os.environ.get("H1R_AMP", "1") == "1"
_H1R_DIV_WEIGHT = float(_os.environ.get("H1R_DIV_WEIGHT", "3.0"))
_H1R_SELECT = _os.environ.get("H1R_SELECT", "acc_f1")
_H1R_COSINE = _os.environ.get("H1R_COSINE", "1") == "1"
_H1R_IGNORE_RADIUS_VOX = int(_os.environ.get("H1R_IGNORE_RADIUS_VOX", "0"))


def _h1r_enable_amp_unet() -> None:
    """Autocast fp16 inside TemporalUNet3D.forward (thread-local => DataParallel-safe).

    Output is cast back to float32 so the detect head, transformer and both losses --
    including F.binary_cross_entropy at compute_loss, which raises under autocast --
    keep their fp32 semantics unchanged.
    """
    if getattr(TemporalUNet3D, "_h1r_amp", False):
        return
    _orig = TemporalUNet3D.forward

    def _fwd(self, x, _orig=_orig):
        if getattr(self, "_h1r_amp_enabled", _H1R_AMP) and torch.cuda.is_available():
            with torch.autocast("cuda", dtype=torch.float16):
                out = _orig(self, x)
            return out.float()
        return _orig(self, x)

    TemporalUNet3D.forward = _fwd
    TemporalUNet3D._h1r_amp = True
    TemporalUNet3D._h1r_amp_enabled = _H1R_AMP


_h1r_enable_amp_unet()
'''

PATCHES: list[tuple[str, str, int]] = [
    # --- knobs + P1 AMP-in-UNet (inserted right after the last vendored import) ------
    (
        "from itertools import cycle as _cycle\n",
        _KNOBS,
        1,
    ),
    # --- P3: real division upweight (F6 no-op fix; trainer :68-70) -------------------
    (
        "    div_rows = target.sum(dim=1) > 1\n"
        "    weight = torch.ones_like(loss)\n"
        "    weight[div_rows] = 1.0\n",
        "    div_rows = target.sum(dim=1) > 1\n"
        "    weight = torch.ones_like(loss)\n"
        "    weight[div_rows] = _H1R_DIV_WEIGHT\n",
        1,
    ),
    # --- P4: sparse-annotation ignore mask (recipe C2; off unless env set) -----------
    (
        "    weight = torch.where(target == 1.0, w_pos, w_neg)\n"
        "\n"
        "    return F.binary_cross_entropy_with_logits(\n",
        "    weight = torch.where(target == 1.0, w_pos, w_neg)\n"
        "\n"
        "    if _H1R_IGNORE_RADIUS_VOX > 0:\n"
        "        # Recipe C2: confident non-GT peaks are likely unannotated true nuclei\n"
        "        # (2.8% of instances labelled) -- ignore a ball around them instead of\n"
        "        # teaching them as background. Enable for the sparse competition\n"
        "        # fine-tune stage only.\n"
        "        with torch.no_grad():\n"
        "            _r = _H1R_IGNORE_RADIUS_VOX\n"
        "            _k = 2 * _r + 1\n"
        "            _lg = logits.unsqueeze(1)\n"
        "            _pooled = F.max_pool3d(_lg, _k, stride=1, padding=_r)\n"
        "            _peaks = (_lg == _pooled) & (_lg > 0.0)\n"
        "            _near_gt = F.max_pool3d(target.unsqueeze(1), _k, stride=1, padding=_r) > 0\n"
        "            _ignore = (_peaks & ~_near_gt).float()\n"
        "            _ball = F.max_pool3d(_ignore, _k, stride=1, padding=_r)[:, 0] > 0\n"
        "        weight = torch.where(_ball & (target != 1.0),\n"
        "                             torch.zeros_like(weight), weight)\n"
        "\n"
        "    return F.binary_cross_entropy_with_logits(\n",
        1,
    ),
    # --- P1: GradScaler in train_epoch ----------------------------------------------
    (
        "    model.train()\n"
        "    total_edge_loss = 0.0\n",
        "    model.train()\n"
        "    scaler = torch.amp.GradScaler(\"cuda\", enabled=_H1R_AMP and torch.cuda.is_available())\n"
        "    total_edge_loss = 0.0\n",
        1,
    ),
    (
        "        optimizer.zero_grad()\n"
        "        loss.backward()\n"
        "        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n"
        "        optimizer.step()\n",
        "        optimizer.zero_grad()\n"
        "        scaler.scale(loss).backward()\n"
        "        scaler.unscale_(optimizer)\n"
        "        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n"
        "        scaler.step(optimizer)\n"
        "        scaler.update()\n",
        1,
    ),
    # --- P2: node precision out of evaluate ------------------------------------------
    (
        "    Returns (avg_loss, accuracy, node_recall).\n",
        "    Returns (avg_loss, accuracy, node_recall, node_precision).\n",
        1,
    ),
    (
        "    total_loss, correct, total, n_pairs = 0.0, 0, 0, 0\n"
        "    gt_matched, gt_total = 0, 0\n",
        "    total_loss, correct, total, n_pairs = 0.0, 0, 0, 0\n"
        "    gt_matched, gt_total = 0, 0\n"
        "    det_matched, det_total = 0, 0\n",
        1,
    ),
    (
        "                gt_total += n_gt\n"
        "                gt_matched += n_matched\n",
        "                gt_total += n_gt\n"
        "                gt_matched += n_matched\n"
        "                det_total += int(matches[b].shape[0])\n"
        "                det_matched += int(n_matched)\n",
        1,
    ),
    (
        "    node_recall = gt_matched / max(gt_total, 1)\n"
        "    return total_loss / max(n_pairs, 1), correct / max(total, 1), node_recall\n",
        "    node_recall = gt_matched / max(gt_total, 1)\n"
        "    node_precision = det_matched / max(det_total, 1)\n"
        "    return (total_loss / max(n_pairs, 1), correct / max(total, 1),\n"
        "            node_recall, node_precision)\n",
        1,
    ),
    # --- P2: precision-aware checkpoint selection (F2 fix; trainer :1177-1180) -------
    (
        "        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um)\n"
        "        test_time = time.monotonic() - t0\n"
        "\n"
        "        score = test_acc * test_recall\n",
        "        test_loss, test_acc, test_recall, test_precision = evaluate(\n"
        "            model, test_loader, device, pool_kernel_um=pool_kernel_um)\n"
        "        test_time = time.monotonic() - t0\n"
        "\n"
        "        if _H1R_SELECT == \"acc_recall\":\n"
        "            score = test_acc * test_recall\n"
        "        else:\n"
        "            _det_f1 = (2.0 * test_precision * test_recall\n"
        "                       / max(test_precision + test_recall, 1e-9))\n"
        "            score = test_acc * _det_f1\n",
        1,
    ),
    (
        "            f\"test_loss={test_loss:.4f} | acc={test_acc:.4f} | recall={test_recall:.4f} | best={best_score:.4f} {marker} | \"\n",
        "            f\"test_loss={test_loss:.4f} | acc={test_acc:.4f} | recall={test_recall:.4f} | \"\n"
        "            f\"prec={test_precision:.4f} | best={best_score:.4f} {marker} | \"\n",
        1,
    ),
    # --- P5: cosine LR over epochs (recipe B4) ---------------------------------------
    (
        "    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)\n"
        "    print(f\"Starting training for {n_epochs} epochs (batch_size={batch_size})...\", flush=True)\n",
        "    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)\n"
        "    scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(n_epochs, 1))\n"
        "                 if _H1R_COSINE else None)\n"
        "    print(f\"Starting training for {n_epochs} epochs (batch_size={batch_size})...\", flush=True)\n",
        1,
    ),
    # --- P6: always-save last checkpoint (session-chain resume) + LR step ------------
    (
        "        marker = \"*\" if is_best else \" \"\n",
        "        torch.save(\n"
        "            {k.replace(\"unet.module.\", \"unet.\", 1): v for k, v in model.state_dict().items()},\n"
        "            output_dir / \"edge_predictor_last.pth\",\n"
        "        )\n"
        "        if scheduler is not None:\n"
        "            scheduler.step()\n"
        "        marker = \"*\" if is_best else \" \"\n",
        1,
    ),
]


def apply_h1r_trainer_patch(trainer_path: Path | str) -> None:
    """Apply all patches to *trainer_path* in place; assert exact counts; compile-check."""
    trainer_path = Path(trainer_path)
    src = trainer_path.read_text(encoding="utf-8")
    if "_H1R_AMP" in src:
        print(f"h1r_trainer_patch: {trainer_path} already patched -- skipping")
        return
    for i, (old, new, expect) in enumerate(PATCHES):
        n = src.count(old)
        assert n == expect, (
            f"h1r_trainer_patch: patch {i} matched {n} times (expected {expect}) "
            f"in {trainer_path}"
        )
        src = src.replace(old, new, expect)
    compile(src, str(trainer_path), "exec")
    trainer_path.write_text(src, encoding="utf-8")
    print(f"h1r_trainer_patch: {len(PATCHES)} patches applied to {trainer_path}")


# NOTE: notebook cells execute with __name__ == "__main__", so the CLI below is guarded
# by the H1R_KERNEL env var (set in the kernel's env cell) to keep cell-embedding inert.
if __name__ == "__main__":
    import os as _os

    if _os.environ.get("H1R_KERNEL") != "1":
        import argparse

        _ap = argparse.ArgumentParser()
        _ap.add_argument("--trainer", required=True,
                         help="path to a COPY of train_unet_transformer.py (patched in place)")
        _a = _ap.parse_args()
        apply_h1r_trainer_patch(_a.trainer)


In [ ]:
r"""H1-R detector-half trainer -- dense-Zebrahub detection training on `kkunizaw/biohub-zh001r`.

Implements recipe B1-Stage-1 for the DETECTOR half only (the packaged nodes carry no track
identity -- verified by `scripts/win_bet/h1r_zh001r_audit.py` -- so the edge head cannot be
supervised here; its weights ride along untouched and the saved checkpoint stays loadable by
the deployed `predict_unet_transformer.load_model`).

Requires `h1r_trainer_patch.apply_h1r_trainer_patch` to have been applied to the
train_unet_transformer.py COPY on sys.path first (AMP inside the UNet forward, GradScaler
pattern; this driver has its own loop but reuses the patched module's primitives:
TemporalUNet3D / UNetNodeTransformer / compute_detection_loss / detect_and_match).

Checkpoint selection is detection F1 on held-out crops at the DEPLOYED operating point
(sigmoid 0.96875 == BIOHUB_DET_THRESHOLD in the P0-B kernel; logit 3.434), which is the A3 fix:
the vendored `acc*recall` score has no precision term and selects the junkiest detector.

The default trunk contract is deliberately conservative: the shared UNet (including BatchNorm
buffers) is frozen and kept in eval mode, so detector-head tuning cannot silently rewrite the
features consumed by the already-trained association transformer.  `adabn_control` is a
diagnostic-only contract that requires LR=0 and permits BatchNorm running-stat adaptation.

All knobs are env-driven so kernel smoke/pilot specs differ only in their env cell:
  H1R_ROOT            dir holding zh001r_iso.npy / zh001r_nodes.npz
  H1R_OUT             output dir (checkpoints + metrics.json)
  H1R_INIT_WEIGHTS    public 50-ep full checkpoint (edge_predictor_best.pth); "" = scratch
  H1R_EPOCHS / H1R_BS / H1R_LR / H1R_NEG_WEIGHT / H1R_MAX_STEPS (0 = full epoch)
  H1R_VAL_MOD         crop_index % H1R_VAL_MOD == 0 -> validation crop (default 6 -> 12/72)
  H1R_AMP             shared with the trainer patch (default 1)
  H1R_ROT90           in-plane 90-degree rotation augmentation (default 1; recipe section 5:
                      grid is isotropic but the PSF is not -- xy rotations only, never z-swaps)
  H1R_RESUME          resume from H1R_OUT/detector_last.pth if present (default 1)

Local CLI (CPU smoke):
  .venv\Scripts\python.exe scripts\kaggle_edits\h1r_det_train.py --trainer-dir <dir-with-patched-trainer> \
      --root <zh001r-dir> --out /tmp/h1r --epochs 1 --max-steps 5
"""
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

ZH_VOX_UM = 1.625          # exact registration geometry; supersedes the old 1.677 um ruler
DEPLOYED_SIGMOID_THR = 0.96875   # BIOHUB_DET_THRESHOLD in the deployed P0-B kernel env cell
POOL_KERNEL_UM = 3.0       # deployed PredictConfig.pool_kernel_um
MATCH_UM = 5.0             # matches trainer detect_and_match default


def _logit(p: float) -> float:
    return float(np.log(p / (1.0 - p)))


def eval_thresholds(raw: str) -> dict[str, float]:
    """Parse probability operating points into named logit thresholds."""
    probs = sorted({float(x.strip()) for x in raw.split(",") if x.strip()})
    if not probs or any(not 0.0 < p < 1.0 for p in probs):
        raise ValueError(f"H1R_EVAL_PROBS must contain probabilities in (0,1), got {raw!r}")
    if DEPLOYED_SIGMOID_THR not in probs:
        probs.append(DEPLOYED_SIGMOID_THR)
        probs.sort()
    return {
        ("deployed" if p == DEPLOYED_SIGMOID_THR else f"p{p:g}"): _logit(p)
        for p in probs
    }


def best_operating_point(val: dict[str, dict]) -> tuple[str, dict]:
    """Diagnostic best point in a sweep; never used for checkpoint promotion."""
    return max(val.items(), key=lambda item: (item[1]["f1"], item[1]["precision"]))


def selection_at_threshold(val: dict[str, dict], name: str = "deployed") -> tuple[str, dict]:
    """Return the predeclared deployment operating point, failing closed if absent."""
    if name not in val:
        raise KeyError(f"selection threshold {name!r} missing from {sorted(val)}")
    return name, val[name]


# =============================================================================
# Data
# =============================================================================

class Zh001rWindows(Dataset):
    """(crop, t0) -> W consecutive frames + padded GT node coords.

    Layout mirrors the trainer's batch dict subset the detector loss needs:
    imgs (W, 64,64,64) float32 quantile-normalised; coords (W, M, 3); masks (W, M).
    Intensity: per-crop 0.1%/99.9% quantile min-max with clamp(0) -- the same robust
    scheme the deployed loader applies to competition uint16 volumes
    (train_unet_transformer.FrameWindowDataset.__getitem__), applied to their uint8.
    """

    def __init__(self, root: Path, crop_ids: list[int], window: int = 2,
                 max_nodes: int | None = None, augment: bool = False,
                 rot90: bool = True):
        self.iso = np.load(Path(root) / "zh001r_iso.npy", mmap_mode="r")
        # Eager materialisation avoids sharing a live ZipFile handle across DataLoader
        # worker processes. The pack is small (~9 MB compressed) relative to the images.
        with np.load(Path(root) / "zh001r_nodes.npz", allow_pickle=False) as node_pack:
            self.nodes = {key: node_pack[key] for key in node_pack.files}
        self.window = window
        self.n_t = int(self.iso.shape[1])
        self.spatial = tuple(int(s) for s in self.iso.shape[2:])
        self.crop_ids = list(crop_ids)
        self.augment = augment
        self.rot90 = rot90
        self.items = [(c, t) for c in self.crop_ids
                      for t in range(self.n_t - window + 1)]
        self.qlo: dict[int, float] = {}
        self.qhi: dict[int, float] = {}
        for c in self.crop_ids:
            v = np.asarray(self.iso[c]).astype(np.float32)
            self.qlo[c] = float(np.quantile(v, 0.001))
            self.qhi[c] = float(np.quantile(v, 0.999))
        if max_nodes is None:
            max_nodes = max(int(self.nodes[f"f{c * self.n_t + t}"].shape[0])
                            for c in self.crop_ids for t in range(self.n_t))
        self.max_nodes = int(max_nodes)

    def __len__(self) -> int:
        return len(self.items)

    def _coords(self, c: int, t: int) -> np.ndarray:
        a = self.nodes[f"f{c * self.n_t + t}"][:, 1:].astype(np.float32)  # z, y, x
        inside = np.all((a >= 0) & (a < np.array(self.spatial)), axis=1)
        a = a[inside]
        return a[: self.max_nodes]

    def __getitem__(self, i: int) -> dict:
        c, t0 = self.items[i]
        W, M = self.window, self.max_nodes
        raw = np.asarray(self.iso[c, t0:t0 + W]).astype(np.float32)
        imgs_np = np.clip((raw - self.qlo[c]) / (self.qhi[c] - self.qlo[c] + 1e-6), 0.0, None)
        imgs = torch.from_numpy(imgs_np)
        coords = torch.zeros(W, M, 3)
        masks = torch.zeros(W, M, dtype=torch.bool)
        for w in range(W):
            a = self._coords(c, t0 + w)
            n = len(a)
            coords[w, :n] = torch.from_numpy(a)
            masks[w, :n] = True

        if self.augment:
            rng = np.random.default_rng()
            imgs = imgs + float(rng.uniform(-0.1, 0.1))            # brightness (vendored)
            flip = rng.random(3) < 0.5                              # 8 axis flips (vendored)
            dims = [1 + d for d, f in enumerate(flip) if f]
            if dims:
                imgs = imgs.flip(dims=dims)
                coords = coords.clone()
                for d in range(3):
                    if flip[d]:
                        coords[masks, d] = imgs.shape[1 + d] - coords[masks, d] - 1
            if self.rot90:
                k = int(rng.integers(0, 4))                         # in-plane 90-degree rots
                if k:
                    imgs = torch.rot90(imgs, k, dims=(2, 3))        # (W, z, y, x): rotate y,x
                    coords = coords.clone()
                    X = self.spatial[2]
                    for _ in range(k):
                        y = coords[..., 1].clone()
                        x = coords[..., 2].clone()
                        coords[..., 1] = X - 1 - x
                        coords[..., 2] = y
        return {"imgs": imgs, "coords": coords, "masks": masks}


def global_max_nodes(root: Path) -> int:
    """Maximum nodes in any frame; a train-only maximum can truncate validation GT."""
    with np.load(Path(root) / "zh001r_nodes.npz", allow_pickle=False) as nodes:
        return max(int(nodes[key].shape[0]) for key in nodes.files)


# =============================================================================
# Train / eval
# =============================================================================

def encode_detection_logits(model, imgs: torch.Tensor, use_tta: bool) -> list[torch.Tensor]:
    """Encode detector logits with the exact deployed eight-view planar transform set."""
    _, det_logits = model.encode(imgs)
    if not use_tta:
        return det_logits

    n_views = 1
    for dims in [(-1,), (-2,), (-2, -1)]:
        _, transformed = model.encode(imgs.flip(dims))
        for f in range(len(det_logits)):
            det_logits[f] = det_logits[f] + transformed[f].flip(dims)
        n_views += 1
    for k in (1, 3):
        _, transformed = model.encode(torch.rot90(imgs, k, dims=(-2, -1)))
        for f in range(len(det_logits)):
            det_logits[f] = det_logits[f] + torch.rot90(
                transformed[f], -k, dims=(-2, -1)
            )
        n_views += 1
    _, transformed = model.encode(imgs.transpose(-1, -2))
    for f in range(len(det_logits)):
        det_logits[f] = det_logits[f] + transformed[f].transpose(-1, -2)
    n_views += 1
    anti = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)
    _, transformed = model.encode(anti)
    for f in range(len(det_logits)):
        det_logits[f] = det_logits[f] + torch.rot90(
            transformed[f].transpose(-1, -2), -1, dims=(-2, -1)
        )
        det_logits[f] = det_logits[f] / (n_views + 1)
    return det_logits


@torch.no_grad()
def eval_detection(T, model, loader, device, thresholds: dict[str, float],
                   use_tta: bool = True) -> dict:
    """Detection P/R/F1 vs zh001r nodes at each logit threshold in *thresholds*.

    Non-overlapping eval windows count every frame exactly once. Matching is the
    trainer's own greedy one-to-one detect_and_match at MATCH_UM physical microns.
    """
    model.eval()
    stats = {k: {"det": 0, "gt": 0, "matched": 0} for k in thresholds}
    for batch in loader:
        imgs = batch["imgs"].to(device, dtype=torch.float32, non_blocking=True)
        coords = batch["coords"].to(device, non_blocking=True)
        masks = batch["masks"].to(device, non_blocking=True)
        B, W = imgs.shape[:2]
        det_logits = encode_detection_logits(model, imgs, use_tta=use_tta)
        for name, thr in thresholds.items():
            for i in range(W):
                _, _, det_m, matches = T.detect_and_match(
                    det_logits[i], coords[:, i], masks[:, i],
                    (W,) + tuple(imgs.shape[2:]),
                    det_threshold=thr,
                    pool_kernel_um=POOL_KERNEL_UM,
                    max_match_distance=MATCH_UM,
                    voxel_size=(ZH_VOX_UM,) * 3,
                    frame_index=i, window_size=W,
                )
                for b in range(B):
                    stats[name]["det"] += int(matches[b].shape[0])
                    stats[name]["gt"] += int(masks[b, i].sum().item())
                    stats[name]["matched"] += int((matches[b] >= 0).sum().item())
    out = {}
    for name, s in stats.items():
        p = s["matched"] / max(s["det"], 1)
        r = s["matched"] / max(s["gt"], 1)
        out[name] = {"precision": p, "recall": r,
                     "f1": 2 * p * r / max(p + r, 1e-9),
                     "n_det": s["det"], "n_gt": s["gt"]}
    return out


def _strip_dp(state: dict) -> dict:
    return {k.replace("unet.module.", "unet.", 1): v for k, v in state.items()}


def optimizer_step_index(optimizer: torch.optim.Optimizer) -> int:
    """Largest per-parameter Adam step; unlike LR, this advances only if AdamW ran."""
    steps: list[int] = []
    for state in optimizer.state.values():
        value = state.get("step")
        if value is not None:
            steps.append(int(value.item() if torch.is_tensor(value) else value))
    return max(steps, default=0)


def _train_step(T, model, batch, device, neg_weight, optimizer, scaler) -> dict:
    imgs = batch["imgs"].to(device, dtype=torch.float32, non_blocking=True)
    coords = batch["coords"].to(device, non_blocking=True)
    masks = batch["masks"].to(device, non_blocking=True)
    B, W = imgs.shape[:2]
    _, det_logits = model.encode(imgs)
    loss = sum(
        T.compute_detection_loss(det_logits[i], coords[:, i], masks[:, i], neg_weight)
        for i in range(W)
    ) / W
    optimizer.zero_grad()
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(
        (p for p in model.parameters() if p.requires_grad), 1.0
    )
    finite_gradients = bool(torch.isfinite(grad_norm).item())
    step_before = optimizer_step_index(optimizer)
    scale_before = float(scaler.get_scale())
    scaler.step(optimizer)
    scaler.update()
    step_after = optimizer_step_index(optimizer)
    scale_after = float(scaler.get_scale())
    return {
        "loss": float(loss.detach()), "batch_size": B,
        "finite_loss": bool(torch.isfinite(loss.detach()).item()),
        "finite_gradients": finite_gradients,
        "grad_norm": float(grad_norm.detach()),
        "optimizer_step_before": step_before,
        "optimizer_step_after": step_after,
        "optimizer_step_occurred": step_after > step_before,
        "scaler_scale_before": scale_before,
        "scaler_scale_after": scale_after,
        "scaler_backoff": scale_after < scale_before,
    }


def configure_trunk_contract(model, contract: str, lr: float) -> None:
    """Apply the explicit shared-representation safety contract."""
    if contract not in {"freeze", "adabn_control", "joint_unsafe"}:
        raise ValueError(f"unknown H1R_TRUNK_CONTRACT={contract!r}")
    if contract == "adabn_control" and lr != 0.0:
        raise ValueError("adabn_control is diagnostic-only and requires H1R_LR=0")
    if contract == "freeze":
        for parameter in model.unet.parameters():
            parameter.requires_grad_(False)


def enforce_trunk_mode(model, contract: str) -> None:
    """`model.train()` also toggles frozen BN modules, so reassert eval mode each epoch."""
    if contract == "freeze":
        model.unet.eval()


def trunk_snapshot(model) -> dict[str, torch.Tensor]:
    return {
        k: v.detach().cpu().clone()
        for k, v in _strip_dp(model.state_dict()).items()
        if k.startswith("unet.")
    }


def assert_trunk_unchanged(model, before: dict[str, torch.Tensor]) -> None:
    after = trunk_snapshot(model)
    if after.keys() != before.keys():
        raise RuntimeError("frozen trunk state keys changed")
    changed = [k for k in before if not torch.equal(before[k], after[k])]
    if changed:
        raise RuntimeError(f"frozen trunk contract violated: {changed[:5]}")


def benchmark_amp(T, model, loader, device, neg_weight, lr, steps: int) -> dict:
    """Paired end-to-end fp32/fp16 timing on the exact DataParallel training path."""
    if device.type != "cuda" or steps <= 0:
        return {"skipped": True, "reason": "no CUDA or H1R_BENCH_STEPS=0"}
    initial = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    rng_cpu = torch.get_rng_state()
    rng_cuda = torch.cuda.get_rng_state_all()
    timings = {}
    for use_amp in (False, True):
        model.load_state_dict(initial)
        T.TemporalUNet3D._h1r_amp_enabled = use_amp
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        iterator = iter(loader)
        model.train()
        torch.cuda.synchronize()
        t0 = time.monotonic()
        done = 0
        while done < steps:
            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                batch = next(iterator)
            _train_step(T, model, batch, device, neg_weight, optimizer, scaler)
            done += 1
        torch.cuda.synchronize()
        timings["fp16" if use_amp else "fp32"] = {
            "steps": done, "seconds": time.monotonic() - t0,
        }
        del optimizer, scaler
    model.load_state_dict(initial)
    torch.set_rng_state(rng_cpu)
    torch.cuda.set_rng_state_all(rng_cuda)
    fp32_s = timings["fp32"]["seconds"]
    fp16_s = timings["fp16"]["seconds"]
    timings["speedup_fp32_over_fp16"] = fp32_s / max(fp16_s, 1e-9)
    return timings


def run_h1r_detector(trainer_dir: Path, root: Path, out_dir: Path) -> dict:
    trainer_dir = Path(trainer_dir)
    # The vendored scripts are normally launched with PYTHONPATH=src by the inference
    # notebook. Importing the trainer in-process must recreate both search roots.
    sys.path.insert(0, str(trainer_dir.parent / "src"))
    sys.path.insert(0, str(trainer_dir))
    import train_unet_transformer as T  # patched copy (h1r_trainer_patch applied)
    assert getattr(T.TemporalUNet3D, "_h1r_amp", False), \
        "apply h1r_trainer_patch to the trainer copy before importing it"

    env = os.environ.get
    epochs = int(env("H1R_EPOCHS", "20"))
    bs = int(env("H1R_BS", "16"))
    lr = float(env("H1R_LR", "1e-4"))
    neg_weight = float(env("H1R_NEG_WEIGHT", "0.1"))   # dense labels: 10x the deployed 1e-2
    max_steps = int(env("H1R_MAX_STEPS", "0"))
    val_mod = int(env("H1R_VAL_MOD", "6"))
    amp = env("H1R_AMP", "1") == "1" and torch.cuda.is_available()
    rot90 = env("H1R_ROT90", "1") == "1"
    resume = env("H1R_RESUME", "1") == "1"
    init_weights = env("H1R_INIT_WEIGHTS", "")
    num_workers = int(env("H1R_WORKERS", "3"))
    seed = int(env("H1R_SEED", "0"))
    bench_steps = int(env("H1R_BENCH_STEPS", "50"))
    min_amp_speedup = float(env("H1R_MIN_AMP_SPEEDUP", "1.3"))
    eval_tta = env("H1R_EVAL_TTA", "1") == "1"
    selection_threshold = env("H1R_SELECT_THRESHOLD", "deployed")
    trunk_contract = env("H1R_TRUNK_CONTRACT", "freeze")
    require_optimizer_steps = env("H1R_REQUIRE_OPT_STEPS", "1") == "1"
    patience = int(env("H1R_PATIENCE", "3"))
    min_delta = float(env("H1R_MIN_DELTA", "1e-4"))
    thresholds = eval_thresholds(env(
        "H1R_EVAL_PROBS", "0.50,0.75,0.90,0.95,0.96875,0.98,0.99"
    ))

    torch.manual_seed(seed)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    # predict_unet_transformer.load_model reads config.json next to the weights;
    # write it so the retrained checkpoint is deploy-loadable as-is.
    (out_dir / "config.json").write_text(json.dumps({
        "unet_out_channels": 32, "unet_layers": [32, 64, 128],
        "downsample": [1, 4, 4], "window_size": 2, "pool_kernel_um": 3.0,
    }, indent=2))

    iso = np.load(Path(root) / "zh001r_iso.npy", mmap_mode="r")
    n_crop = int(iso.shape[0])
    val_ids = [c for c in range(n_crop) if c % val_mod == 0]
    train_ids = [c for c in range(n_crop) if c % val_mod != 0]
    print(f"zh001r: {n_crop} crops -> {len(train_ids)} train / {len(val_ids)} val "
          f"(val = idx % {val_mod} == 0)", flush=True)

    max_nodes = global_max_nodes(root)
    train_ds = Zh001rWindows(root, train_ids, max_nodes=max_nodes,
                             augment=True, rot90=rot90)
    val_ds = Zh001rWindows(root, val_ids, augment=False, max_nodes=max_nodes)
    # Non-overlapping val windows: stride = window so each frame is counted once.
    val_ds.items = [(c, t) for c in val_ids for t in range(0, val_ds.n_t - 1, 2)]
    print(f"windows: train={len(train_ds)} val={len(val_ds)} max_nodes={train_ds.max_nodes}",
          flush=True)

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                              num_workers=num_workers, pin_memory=False,
                              persistent_workers=num_workers > 0)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False,
                            num_workers=num_workers, pin_memory=False,
                            persistent_workers=num_workers > 0)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    unet = T.TemporalUNet3D(in_channels=1, out_channels=32, layers=[32, 64, 128])
    model = T.UNetNodeTransformer(unet=unet, unet_out_channels=32,
                                  pos_feat_dim=4 * T._POS_EMBED_DIM)
    if init_weights:
        state = torch.load(init_weights, map_location="cpu", weights_only=True)
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"init from {init_weights}: {len(missing)} missing, "
              f"{len(unexpected)} unexpected", flush=True)
        if missing or unexpected:
            raise RuntimeError({"missing": missing, "unexpected": unexpected})
    model.to(device)
    configure_trunk_contract(model, trunk_contract, lr)

    start_epoch = 0
    history: list[dict] = []
    last_path = out_dir / "detector_last.pth"
    resume_ck = None
    if resume and last_path.exists():
        resume_ck = torch.load(last_path, map_location=device, weights_only=True)
        model.load_state_dict(resume_ck["model"])
        start_epoch = int(resume_ck["epoch"]) + 1
        history = json.loads((out_dir / "metrics.json").read_text()) \
            if (out_dir / "metrics.json").exists() else []
        print(f"resumed at epoch {start_epoch} from {last_path}", flush=True)

    if device.type == "cuda" and torch.cuda.device_count() > 1:
        model.unet = torch.nn.DataParallel(model.unet)
        print(f"DataParallel UNet across {torch.cuda.device_count()} GPUs", flush=True)
    # Snapshot after a possible resume load and DataParallel wrapping. The normalised
    # state keys still match deployment, and the guard now checks the state actually trained.
    frozen_trunk = trunk_snapshot(model) if trunk_contract == "freeze" else None

    trainable = [p for p in model.parameters() if p.requires_grad]
    if not trainable:
        raise RuntimeError("trunk contract left no trainable parameters")
    opt = torch.optim.AdamW(trainable, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    if resume_ck is not None and "optimizer" in resume_ck:
        opt.load_state_dict(resume_ck["optimizer"])
        sched.load_state_dict(resume_ck["scheduler"])
        scaler.load_state_dict(resume_ck["scaler"])
        torch.set_rng_state(resume_ck["rng_cpu"].cpu())
        if device.type == "cuda" and resume_ck.get("rng_cuda"):
            torch.cuda.set_rng_state_all(resume_ck["rng_cuda"])
    else:
        for _ in range(start_epoch):
            sched.step()

    if start_epoch == 0:
        bench = benchmark_amp(T, model, train_loader, device, neg_weight, lr, bench_steps)
        T.TemporalUNet3D._h1r_amp_enabled = amp
        (out_dir / "amp_benchmark.json").write_text(json.dumps(bench, indent=2))
        print(f"AMP BENCHMARK: {json.dumps(bench, sort_keys=True)}", flush=True)
        if amp and not bench.get("skipped") \
                and bench["speedup_fp32_over_fp16"] < min_amp_speedup:
            raise RuntimeError(
                f"AMP speedup {bench['speedup_fp32_over_fp16']:.3f}x is below "
                f"H1R_MIN_AMP_SPEEDUP={min_amp_speedup:.3f}; aborting before full training")

    # The zero-epoch checkpoint is the scientific baseline and a safe fallback if every
    # fine-tuned epoch is worse. It is measured across a real operating-point sweep.
    if not history:
        baseline_val = eval_detection(
            T, model, val_loader, device, thresholds, use_tta=eval_tta
        )
        baseline_name, baseline_best = selection_at_threshold(
            baseline_val, selection_threshold
        )
        baseline = {"epoch": -1, "stage": "zero_epoch", "val": baseline_val,
                    "selection": {"threshold": baseline_name, **baseline_best}}
        history.append(baseline)
        torch.save(_strip_dp(model.state_dict()), out_dir / "edge_predictor_best.pth")
        (out_dir / "metrics.json").write_text(json.dumps(history, indent=1))
        print(f"zero epoch | best={baseline_name} P={baseline_best['precision']:.3f} "
              f"R={baseline_best['recall']:.3f} F1={baseline_best['f1']:.3f}", flush=True)

    best_f1 = max(
        (selection_at_threshold(h["val"], selection_threshold)[1]["f1"] for h in history),
        default=0.0,
    )
    stale_epochs = 0
    total_attempted_steps = 0
    total_optimizer_steps = 0
    total_finite_optimizer_steps = 0
    total_nonfinite_steps = 0
    epoch_audits: list[dict] = []
    for epoch in range(start_epoch, epochs):
        model.train()
        enforce_trunk_mode(model, trunk_contract)
        t0 = time.monotonic()
        tot, n = 0.0, 0
        epoch_step_records = []
        for step, batch in enumerate(train_loader):
            if max_steps and step >= max_steps:
                break
            step_record = _train_step(
                T, model, batch, device, neg_weight, opt, scaler
            )
            epoch_step_records.append(step_record)
            tot += step_record["loss"] * step_record["batch_size"]
            n += step_record["batch_size"]
        train_time = time.monotonic() - t0

        attempted = len(epoch_step_records)
        optimizer_steps = sum(r["optimizer_step_occurred"] for r in epoch_step_records)
        finite_optimizer_steps = sum(
            r["optimizer_step_occurred"] and r["finite_loss"] and r["finite_gradients"]
            for r in epoch_step_records
        )
        nonfinite = sum(not (r["finite_loss"] and r["finite_gradients"])
                        for r in epoch_step_records)
        total_attempted_steps += attempted
        total_optimizer_steps += optimizer_steps
        total_finite_optimizer_steps += finite_optimizer_steps
        total_nonfinite_steps += nonfinite
        audit = {
            "epoch": epoch, "attempted_steps": attempted,
            "optimizer_steps": optimizer_steps, "skipped_steps": attempted - optimizer_steps,
            "finite_optimizer_steps": finite_optimizer_steps,
            "nonfinite_steps": nonfinite,
            "final_optimizer_step_index": optimizer_step_index(opt),
            "initial_scaler_scale": (epoch_step_records[0]["scaler_scale_before"]
                                     if epoch_step_records else float(scaler.get_scale())),
            "final_scaler_scale": float(scaler.get_scale()),
            "max_grad_norm": max((r["grad_norm"] for r in epoch_step_records), default=0.0),
        }
        epoch_audits.append(audit)
        training_audit = {
            "contract": trunk_contract, "lr": lr, "eval_tta": eval_tta,
            "selection_threshold": selection_threshold,
            "attempted_steps": total_attempted_steps,
            "optimizer_steps": total_optimizer_steps,
            "finite_optimizer_steps": total_finite_optimizer_steps,
            "skipped_steps": total_attempted_steps - total_optimizer_steps,
            "nonfinite_steps": total_nonfinite_steps,
            "epochs": epoch_audits,
        }
        (out_dir / "training_audit.json").write_text(json.dumps(training_audit, indent=2))
        if require_optimizer_steps and attempted and finite_optimizer_steps == 0:
            raise RuntimeError(
                f"optimizer-step contract failed in epoch {epoch}: "
                f"0/{attempted} finite AdamW steps; "
                f"audit={out_dir / 'training_audit.json'}"
            )

        t0 = time.monotonic()
        val = eval_detection(T, model, val_loader, device, thresholds, use_tta=eval_tta)
        val_time = time.monotonic() - t0
        sched.step()

        d = val["deployed"]
        selected_name, selected = selection_at_threshold(val, selection_threshold)
        is_best = selected["f1"] > best_f1 + min_delta
        state = _strip_dp(model.state_dict())
        if frozen_trunk is not None:
            assert_trunk_unchanged(model, frozen_trunk)
        torch.save({
            "model": state, "epoch": epoch, "optimizer": opt.state_dict(),
            "scheduler": sched.state_dict(), "scaler": scaler.state_dict(),
            "rng_cpu": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],
        }, last_path)
        if is_best:
            best_f1 = selected["f1"]
            torch.save(state, out_dir / "edge_predictor_best.pth")  # deploy-compatible name
            stale_epochs = 0
        else:
            stale_epochs += 1
        history.append({"epoch": epoch, "det_loss": tot / max(n, 1), "val": val,
                        "selection": {"threshold": selected_name, **selected},
                        "lr": sched.get_last_lr()[0],
                        "step_audit": audit,
                        "train_s": train_time, "val_s": val_time})
        (out_dir / "metrics.json").write_text(json.dumps(history, indent=1))
        print(f"epoch {epoch:3d} | det_loss={tot / max(n, 1):.4f} | "
              f"deployed-thr P={d['precision']:.3f} R={d['recall']:.3f} F1={d['f1']:.3f} "
              f"(n_det={d['n_det']} n_gt={d['n_gt']}) | "
              f"selected={selected_name} F1={selected['f1']:.3f} | "
              f"best={best_f1:.3f}{' *' if is_best else ''} | "
              f"train={train_time:.0f}s val={val_time:.0f}s", flush=True)
        if patience > 0 and stale_epochs >= patience:
            print(f"early stop after {stale_epochs} non-improving epochs", flush=True)
            break
    baseline_name, baseline_best = selection_at_threshold(
        history[0]["val"], selection_threshold
    )
    diagnostic_name, diagnostic_best = best_operating_point(history[-1]["val"])
    result = {
        "baseline": {"threshold": baseline_name, **baseline_best},
        "selection_threshold": selection_threshold,
        "best_f1_at_selection_threshold": best_f1,
        "diagnostic_best_final_sweep": {"threshold": diagnostic_name, **diagnostic_best},
        "improvement_over_baseline": best_f1 - baseline_best["f1"],
        "epochs_run": sum(h.get("stage") != "zero_epoch" for h in history),
        "trunk_contract": trunk_contract, "eval_tta": eval_tta,
        "training_audit": training_audit if epoch_audits else {
            "attempted_steps": 0, "optimizer_steps": 0,
        },
        "history": history,
    }
    (out_dir / "summary.json").write_text(json.dumps(result, indent=2))
    return result


if __name__ == "__main__":
    if os.environ.get("H1R_KERNEL") != "1":   # notebook cells run as __main__; stay inert
        import argparse

        ap = argparse.ArgumentParser()
        ap.add_argument("--trainer-dir", required=True,
                        help="dir holding the PATCHED train_unet_transformer.py copy")
        ap.add_argument("--root", required=True)
        ap.add_argument("--out", required=True)
        ap.add_argument("--epochs", type=int, default=None)
        ap.add_argument("--max-steps", type=int, default=None)
        ap.add_argument("--bs", type=int, default=None)
        a = ap.parse_args()
        if a.epochs is not None:
            os.environ["H1R_EPOCHS"] = str(a.epochs)
        if a.max_steps is not None:
            os.environ["H1R_MAX_STEPS"] = str(a.max_steps)
        if a.bs is not None:
            os.environ["H1R_BS"] = str(a.bs)
        run_h1r_detector(Path(a.trainer_dir), Path(a.root), Path(a.out))


In [ ]:
"""Final cell for the H1-R detector training kernels.

The patch and driver are embedded in earlier notebook cells. This cell resolves mounted
datasets without assuming Kaggle's directory layout, applies the trainer patch to the
ephemeral repository copy, runs S1, and exposes the promotion artifacts at output root.
"""
from __future__ import annotations

import json
import os
import shutil
from pathlib import Path

import torch


if not torch.cuda.is_available():
    raise RuntimeError("H1-R requires a Kaggle GPU; refusing a silent CPU training run")
print("CUDA:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


def _unique_parent(name: str) -> Path:
    hits = sorted(Path("/kaggle/input").rglob(name))
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {name!r} under /kaggle/input, got {hits}")
    return hits[0].parent


zh_root = _unique_parent("zh001r_iso.npy")
nodes_root = _unique_parent("zh001r_nodes.npz")
if nodes_root != zh_root:
    raise RuntimeError({"iso_root": str(zh_root), "nodes_root": str(nodes_root)})

trainer_path = REPO_DIR / "scripts" / "train_unet_transformer.py"
init_weights = REPO_DIR / "weights" / "unet_transformer" / "split_0" / "edge_predictor_best.pth"
for required in (trainer_path, init_weights, zh_root / "zh001r_nodes.npz"):
    if not required.is_file():
        raise FileNotFoundError(required)

apply_h1r_trainer_patch(trainer_path)
os.environ["H1R_ROOT"] = str(zh_root)
os.environ["H1R_INIT_WEIGHTS"] = str(init_weights)
os.environ["H1R_OUT"] = "/kaggle/working/h1r_detector"

result = run_h1r_detector(
    trainer_dir=trainer_path.parent,
    root=zh_root,
    out_dir=Path(os.environ["H1R_OUT"]),
)

out = Path(os.environ["H1R_OUT"])
for name in ("edge_predictor_best.pth", "detector_last.pth", "config.json",
             "metrics.json", "summary.json", "amp_benchmark.json",
             "training_audit.json"):
    source = out / name
    if source.is_file():
        shutil.copy2(source, Path("/kaggle/working") / name)

print("H1R_RESULT", json.dumps(result, sort_keys=True))
